In [1]:
import joblib
import os
import numpy as np

print("--- Step 1: Loading & Reshaping for Convolutional Neural Network ---")

processed_dir = "../data/processed/"

# 1. Load the matrices
X_train_smote = joblib.load(os.path.join(processed_dir, "ton_iot_X_train_smote.pkl"))
y_train_smote = joblib.load(os.path.join(processed_dir, "ton_iot_y_train_smote.pkl"))
X_test = joblib.load(os.path.join(processed_dir, "ton_iot_X_test.pkl"))
y_test = joblib.load(os.path.join(processed_dir, "ton_iot_y_test.pkl"))

# 2. Reshape into 3D Tensors: (Samples, TimeSteps/Features, Channels)
# We convert them to NumPy arrays first to ensure perfect mathematical formatting
X_train_cnn = np.array(X_train_smote).reshape(X_train_smote.shape[0], X_train_smote.shape[1], 1)
X_test_cnn = np.array(X_test).reshape(X_test.shape[0], X_test.shape[1], 1)

# Ensure labels are also clean numpy arrays
y_train_cnn = np.array(y_train_smote)
y_test_cnn = np.array(y_test)

print("\nTensor Transformation Complete!")
print(f"CNN Training Tensor Shape: {X_train_cnn.shape}")
print(f"CNN Testing Tensor Shape:  {X_test_cnn.shape}")

--- Step 1: Loading & Reshaping for Convolutional Neural Network ---

Tensor Transformation Complete!
CNN Training Tensor Shape: (1266114, 28, 1)
CNN Testing Tensor Shape:  (199965, 28, 1)


In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

print("--- Step 2: Compiling the SENTRi-X CNN Architecture ---")

# Define the input shape (28 features, 1 channel)
input_shape = (X_train_cnn.shape[1], 1)

# Initialize a Sequential Neural Network
cnn_model = Sequential()

# Layer 1: The First Convolutional Scan (Feature Extraction)
cnn_model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
cnn_model.add(MaxPooling1D(pool_size=2))

# Layer 2: Deeper Pattern Recognition
cnn_model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
cnn_model.add(MaxPooling1D(pool_size=2))

# Layer 3: Flatten the 3D data back to 1D so we can make a final decision
cnn_model.add(Flatten())

# Layer 4: The Decision Layers (Fully Connected)
cnn_model.add(Dense(128, activation='relu'))
cnn_model.add(Dropout(0.5)) # Randomly turn off 50% of neurons to force robust learning
cnn_model.add(Dense(1, activation='sigmoid')) # Binary classification: 0 (Normal) or 1 (Attack)

# Compile the engine
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\nArchitecture Successfully Compiled. Here is the blueprint:")
cnn_model.summary()

--- Step 2: Compiling the SENTRi-X CNN Architecture ---


c:\Users\LENOVO\SENTRi-X\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Architecture Successfully Compiled. Here is the blueprint:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 26, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 13, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 11, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 5, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 640)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 107,137 (418.50 KB)

 Trainable params: 107,137 (418.50 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
import time

print("--- Step 3: Training the CNN (Starting 5 Epochs) ---")

start_time = time.time()

# Train the network!
history = cnn_model.fit(
    X_train_cnn, 
    y_train_cnn, 
    epochs=5, 
    batch_size=512, 
    validation_split=0.2, # Uses 20% of the training data as a mid-training sanity check
    verbose=1
)

end_time = time.time()
print(f"\nCNN Training Complete! Elapsed Time: {round(end_time - start_time, 2)} seconds")

--- Step 3: Training the CNN (Starting 5 Epochs) ---
Epoch 1/5
1979/1979 ━━━━━━━━━━━━━━━━━━━━ 64s 31ms/step - accuracy: 0.9855 - loss: 0.0632 - val_accuracy: 0.9973 - val_loss: 0.0455
Epoch 2/5
1979/1979 ━━━━━━━━━━━━━━━━━━━━ 60s 30ms/step - accuracy: 0.9863 - loss: 0.0543 - val_accuracy: 0.9966 - val_loss: 0.0423
Epoch 3/5
1979/1979 ━━━━━━━━━━━━━━━━━━━━ 60s 30ms/step - accuracy: 0.9864 - loss: 0.0510 - val_accuracy: 0.9978 - val_loss: 0.0382
Epoch 4/5
1979/1979 ━━━━━━━━━━━━━━━━━━━━ 61s 31ms/step - accuracy: 0.9880 - loss: 0.0364 - val_accuracy: 0.9982 - val_loss: 0.0166
Epoch 5/5
1979/1979 ━━━━━━━━━━━━━━━━━━━━ 64s 32ms/step - accuracy: 0.9942 - loss: 0.0217 - val_accuracy: 0.9971 - val_loss: 0.0141

CNN Training Complete! Elapsed Time: 511.92 seconds


In [4]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

print("--- Step 4: Final CNN Evaluation (Unseen Test Set) ---")

# 1. Run the inference
start_inference = time.time()
# The model outputs probabilities. We convert anything > 0.5 into a 1 (Attack)
y_pred_prob = cnn_model.predict(X_test_cnn)
y_pred_cnn = (y_pred_prob > 0.5).astype("int32")
end_inference = time.time()

print(f"\nInference Time for {X_test_cnn.shape[0]:,} rows: {round(end_inference - start_inference, 4)} seconds")
print(f"Accuracy Score: {accuracy_score(y_test_cnn, y_pred_cnn) * 100:.2f}%\n")

# 2. Classification Report
print("Classification Report:")
print(classification_report(y_test_cnn, y_pred_cnn, target_names=['Normal (0)', 'Attack (1)'], digits=4))

# 3. Confusion Matrix
cm_cnn = confusion_matrix(y_test_cnn, y_pred_cnn)
print("\nConfusion Matrix Breakdown:")
print(f"True Negatives (Correctly identified Normal): {cm_cnn[0][0]:,}")
print(f"False Positives (False Alarms):               {cm_cnn[0][1]:,}")
print(f"False Negatives (Missed Attacks):             {cm_cnn[1][0]:,}")
print(f"True Positives (Correctly identified Attack): {cm_cnn[1][1]:,}")

# 4. Save the deep learning model (using .h5 format for Keras)
models_dir = "../models/"
model_path = os.path.join(models_dir, "cnn_model_ton_iot.h5")
cnn_model.save(model_path)

print(f"\nCNN model successfully saved to: {model_path}")

--- Step 4: Final CNN Evaluation (Unseen Test Set) ---
6249/6249 ━━━━━━━━━━━━━━━━━━━━ 24s 4ms/step



Inference Time for 199,965 rows: 28.0893 seconds
Accuracy Score: 99.43%

Classification Report:
              precision    recall  f1-score   support

  Normal (0)     0.9768    0.9964    0.9865     41701
  Attack (1)     0.9990    0.9938    0.9964    158264

    accuracy                         0.9943    199965
   macro avg     0.9879    0.9951    0.9914    199965
weighted avg     0.9944    0.9943    0.9943    199965


Confusion Matrix Breakdown:
True Negatives (Correctly identified Normal): 41,550
False Positives (False Alarms):               151
False Negatives (Missed Attacks):             987
True Positives (Correctly identified Attack): 157,277

CNN model successfully saved to: ../models/cnn_model_ton_iot.h5
